### training scheme
#### (1) train denoising auto encoder model using all data including train and test data
#### (2) from the weights of denoising auto encoder model, finetune to predict targets such as reactivity

### rough network architecture
#### inputs -> conv1ds -> aggregation of neighborhoods -> multi head attention -> aggregation of neighborhoods -> multi head attention -> conv1d -> predict
#### this architecture was inspired by https://www.kaggle.com/cpmpml/graph-transfomer


In [1]:
pretrain_dir = None # model dir for resuming training. if None, train from scrach

one_fold = False # if True, train model at only first fold. use if you try a new idea quickly.
run_test = False # if True, use small data. you can check whether this code run or not
denoise = True # if True, use train data whose signal_to_noise > 1

ae_epochs = 20 # epoch of training of denoising auto encoder
ae_epochs_each = 5 # epoch of training of denoising auto encoder each time. 
                   # I use train data (seqlen = 107) and private test data (seqlen = 130) for auto encoder training.
                   # I dont know how to easily fit keras model to use both of different shape data simultaneously, 
                   # so I call fit function several times. 
ae_batch_size = 32

epochs_list = [30, 10, 3, 3, 5, 5]
batch_size_list = [8, 16, 32, 64, 128, 256] 

## copy pretrain model to working dir
import shutil
import glob
if pretrain_dir is not None:
    for d in glob.glob(pretrain_dir + "*"):
        shutil.copy(d, ".")
    
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import gc
import os
import matplotlib.pyplot as plt
%matplotlib inline

## load

In [2]:
from pathlib import Path

# Supports launching Jupyter either from the project root or from comparaison/.
DATA_DIR = next(path for path in (Path("comparaison/data"), Path("data")) if (path / "train.npz").exists())
TARGETS = ["reactivity", "deg_Mg_pH10", "deg_pH10", "deg_Mg_50C", "deg_50C"]

def load_npz_frame(path, labeled=False):
    with np.load(path, allow_pickle=False) as z:
        ids = z["ids"].astype(str)
        tokens = z["tokens"]
        # These vocabularies are the same as src/config.py in the project.
        seq_vocab, struct_vocab, loop_vocab = "ACGU", "().", "SMIBHEX"
        frame = pd.DataFrame({
            "id": ids,
            "sequence": ["".join(seq_vocab[j] for j in row[:, 0]) for row in tokens],
            "structure": ["".join(struct_vocab[j - 4] for j in row[:, 1]) for row in tokens],
            "predicted_loop_type": ["".join(loop_vocab[j - 7] for j in row[:, 2]) for row in tokens],
            "seq_length": tokens.shape[1],
            "seq_scored": 68,
        })
        if labeled:
            for i, target in enumerate(TARGETS):
                frame[target] = [row.tolist() for row in z["targets"][:, :, i]]
            frame["signal_to_noise"] = z["signal_to_noise"]
        return frame, z["bpps"].copy()

# train.npz contains the 2400 labeled, length-107 molecules.
train, As = load_npz_frame(DATA_DIR / "train.npz", labeled=True)
# These are the 629 public length-107 and 3005 private length-130 test molecules.
test_pub, As_pub = load_npz_frame(DATA_DIR / "test_107.npz")
test_pri, As_pri = load_npz_frame(DATA_DIR / "test_130.npz")
if denoise:
    keep = train.signal_to_noise > 1
    train = train.loc[keep].reset_index(drop=True)
    As = As[keep.to_numpy()]

targets = TARGETS
sub = pd.DataFrame(columns=["id_seqpos", *targets])

if run_test:  # small smoke run only; it is not suitable for five-fold CV
    train, As = train.iloc[:30].reset_index(drop=True), As[:30]
    test_pub, As_pub = test_pub.iloc[:30].reset_index(drop=True), As_pub[:30]
    test_pri, As_pri = test_pri.iloc[:30].reset_index(drop=True), As_pri[:30]


In [3]:
print(train.shape)
train.head()

(2096, 12)


,id,sequence,structure,predicted_loop_type,seq_length,seq_scored,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C,signal_to_noise
0,id_001f94081,GGAAAAGCUCUAAUAACAGGAGACUAGGACUACGUAUUUCUAGGUA...,.....((((((.......)))).)).((.....((..((((((......,EEEEESSSSSSHHHHHHHSSSSBSSXSSIIIIISSIISSSSSSHHH...,107,68,"[0.3296999931335449, 1.5693000555038452, 1.122...","[0.7555999755859375, 2.9830000400543213, 0.252...","[2.3375000953674316, 3.50600004196167, 0.30079...","[0.3580999970436096, 2.9683001041412354, 0.258...","[0.6381999850273132, 3.477299928665161, 0.9987...",6.894
1,id_006f36f57,GGAAAGUGCUCAGAUAAGCUAAGCUCGAAUAGCAAUCGAAUAGAAU...,.....((((.((.....((((.(((.....)))..((((......)...,EEEEESSSSISSIIIIISSSSMSSSHHHHHSSSMMSSSSHHHHHHS...,107,68,"[0.448199987411499, 1.482200026512146, 1.18190...","[0.25040000677108765, 1.4020999670028687, 0.98...","[2.243000030517578, 2.9361000061035156, 1.0552...","[0.5163000226020813, 1.6822999715805054, 1.042...","[0.9501000046730042, 1.7975000143051147, 1.499...",8.800
2,id_00ab2d761,GGAAAGCGCCGCGGCGGUAGCGGCAGCGAGGAGCGCUACCAAGGCA...,.....(.(((((.(((((((((...........)))))))..(((....,EEEEESISSSSSISSSSSSSSSHHHHHHHHHHHSSSSSSSMMSSSH...,107,68,"[0.76419997215271, 1.6641000509262085, 1.06219...","[0.9559000134468079, 1.944200038909912, 1.0113...","[1.955399990081787, 2.1298000812530518, 1.0403...","[0.22460000216960907, 1.7280999422073364, 1.38...","[0.5881999731063843, 1.1785999536514282, 0.970...",4.136
3,id_00abef1d7,GGAAAACAAUUGCAUCGUUAGUACGACUCCACAGCGUAAGCUGUGG...,.........((((((((......((((((((((((....)))))))...,EEEEEEEEESSSSSSSSIIIIIISSSSSSSSSSSSHHHHSSSSSSS...,107,68,"[0.8949999809265137, 2.3376998901367188, 2.230...","[0.4603999853134155, 3.6695001125335693, 0.785...","[2.7711000442504883, 7.364999771118164, 1.6923...","[1.0729999542236328, 2.8603999614715576, 1.993...","[2.096400022506714, 3.368799924850464, 0.63990...",2.485
4,id_00b436dec,GGAAAUCAUCGAGGACGGGUCCGUUCAGCACGCGAAAGCGUCGUGA...,.....(((((((((((..(((((((((..((((....))))..)))...,EEEEESSSSSSSSSSSIISSSSSSSSSIISSSSHHHHSSSSIISSS...,107,68,"[1.1576000452041626, 1.513700008392334, 1.3381...","[1.6912000179290771, 5.265200138092041, 2.3901...","[1.8640999794006348, 2.376699924468994, 1.1490...","[0.49059998989105225, 4.633900165557861, 1.958...","[1.2851999998092651, 2.5460000038146973, 0.234...",1.727


In [4]:
print("public test (length 107):", test_pub.shape)
test_pub.head()
print("private test (length 130):", test_pri.shape)
test_pri.head()

public test (length 107): (629, 6)
private test (length 130): (3005, 6)


,id,sequence,structure,predicted_loop_type,seq_length,seq_scored
0,id_000ae4237,GGAAACGGGUUCCGCGGAUUGCUGCUAAUAAGAGUAAUCUCUAAAU...,.....((((..((((((...(((((.....((((....)))).......,EEEEESSSSIISSSSSSIIISSSSSIIIIISSSSHHHHSSSSIIII...,130,68
1,id_0020473f7,GGAAACCCGCCCGCGCCCGCCCGCGCUGCUGCCGUGCCUCCUCUCC...,.....(((((((((((((((((((((((((((((((((((((((((...,EEEEESSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSSS...,130,68
2,id_002852873,GGAAAGCGAAACGCCGAGAAGACGUAGUUCGCAGAGCGGCGUACCC...,.....(((...(((......(((((((((.((....(((.....))...,EEEEESSSIIISSSBBBBBBSSSSSSSSSBSSBBBBSSSHHHHHSS...,130,68
3,id_0031191b7,GGAAAUGUCUACAUAGGAGUGCUGCGGGACGGUAACGUCAUGACCG...,........(((((((((((.((..(((((((....))))....)))...,EEEEEEEESSSSSSSSSSSISSIISSSSSSSHHHHSSSSBBBBSSS...,130,68
4,id_003ab2445,GGAAAGACUCAGAGGUGAAGGUCAUCACGGCUGAUAGGAGACUAUC...,.....(((((..........((((((.((.(((((((....)))))...,EEEEESSSSSBBBBBBBBBBSSSSSSISSISSSSSSSHHHHSSSSS...,130,68


In [5]:
print(sub.shape)
sub.head()

(0, 6)


,id_seqpos,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C


## target

In [6]:
targets = list(sub.columns[1:])
print(targets)

y_train = []
seq_len = train["seq_length"].iloc[0]
seq_len_target = train["seq_scored"].iloc[0]
ignore = -10000
ignore_length = seq_len - seq_len_target
for target in targets:
    y = np.vstack(train[target])
    dummy = np.zeros([y.shape[0], ignore_length]) + ignore
    y = np.hstack([y, dummy])
    y_train.append(y)
y = np.stack(y_train, axis = 2)
y.shape

['reactivity', 'deg_Mg_pH10', 'deg_pH10', 'deg_Mg_50C', 'deg_50C']


(2096, 107, 5)

## structure adj

In [7]:
from tqdm import tqdm

def get_structure_adj(train):
    ## get adjacent matrix from structure sequence
    
    ## here I calculate adjacent matrix of each base pair, 
    ## but eventually ignore difference of base pair and integrate into one matrix
    Ss = []
    for i in tqdm(range(len(train))):
        seq_length = train["seq_length"].iloc[i]
        structure = train["structure"].iloc[i]
        sequence = train["sequence"].iloc[i]

        cue = []
        a_structures = {
            ("A", "U") : np.zeros([seq_length, seq_length]),
            ("C", "G") : np.zeros([seq_length, seq_length]),
            ("U", "G") : np.zeros([seq_length, seq_length]),
            ("U", "A") : np.zeros([seq_length, seq_length]),
            ("G", "C") : np.zeros([seq_length, seq_length]),
            ("G", "U") : np.zeros([seq_length, seq_length]),
        }
        a_structure = np.zeros([seq_length, seq_length])
        for i in range(seq_length):
            if structure[i] == "(":
                cue.append(i)
            elif structure[i] == ")":
                start = cue.pop()
#                 a_structure[start, i] = 1
#                 a_structure[i, start] = 1
                a_structures[(sequence[start], sequence[i])][start, i] = 1
                a_structures[(sequence[i], sequence[start])][i, start] = 1
        
        a_strc = np.stack([a for a in a_structures.values()], axis = 2)
        a_strc = np.sum(a_strc, axis = 2, keepdims = True)
        Ss.append(a_strc)
    
    Ss = np.array(Ss)
    print(Ss.shape)
    return Ss
Ss = get_structure_adj(train)
Ss_pub = get_structure_adj(test_pub)
Ss_pri = get_structure_adj(test_pri)

100%|██████████| 2096/2096 [00:00<00:00, 4452.30it/s]


(2096, 107, 107, 1)


100%|██████████| 629/629 [00:00<00:00, 4123.64it/s]


(629, 107, 107, 1)


100%|██████████| 3005/3005 [00:00<00:00, 3066.10it/s]


(3005, 130, 130, 1)


## distance adj

In [8]:
def get_distance_matrix(As):
    ## adjacent matrix based on distance on the sequence
    ## D[i, j] = 1 / (abs(i - j) + 1) ** pow, pow = 1, 2, 4
    
    idx = np.arange(As.shape[1])
    Ds = []
    for i in range(len(idx)):
        d = np.abs(idx[i] - idx)
        Ds.append(d)

    Ds = np.array(Ds) + 1
    Ds = 1/Ds
    Ds = Ds[None, :,:]
    Ds = np.repeat(Ds, len(As), axis = 0)
    
    Dss = []
    for i in [1, 2, 4]: 
        Dss.append(Ds ** i)
    Ds = np.stack(Dss, axis = 3)
    print(Ds.shape)
    return Ds

Ds = get_distance_matrix(As)
Ds_pub = get_distance_matrix(As_pub)
Ds_pri = get_distance_matrix(As_pri)

(2096, 107, 107, 3)
(629, 107, 107, 3)
(3005, 130, 130, 3)


In [9]:
## concat adjecent
As = np.concatenate([As[:,:,:,None], Ss, Ds], axis = 3).astype(np.float32)
As_pub = np.concatenate([As_pub[:,:,:,None], Ss_pub, Ds_pub], axis = 3).astype(np.float32)
As_pri = np.concatenate([As_pri[:,:,:,None], Ss_pri, Ds_pri], axis = 3).astype(np.float32)
del Ss, Ds, Ss_pub, Ds_pub, Ss_pri, Ds_pri
As.shape, As_pub.shape, As_pri.shape

((2096, 107, 107, 5), (629, 107, 107, 5), (3005, 130, 130, 5))

## node

In [10]:
## sequence
def return_ohe(n, i):
    tmp = [0] * n
    tmp[i] = 1
    return tmp

def get_input(train):
    ## get node features, which is one hot encoded
    mapping = {}
    vocab = ["A", "G", "C", "U"]
    for i, s in enumerate(vocab):
        mapping[s] = return_ohe(len(vocab), i)
    X_node = np.stack(train["sequence"].apply(lambda x : list(map(lambda y : mapping[y], list(x)))))

    mapping = {}
    vocab = ["S", "M", "I", "B", "H", "E", "X"]
    for i, s in enumerate(vocab):
        mapping[s] = return_ohe(len(vocab), i)
    X_loop = np.stack(train["predicted_loop_type"].apply(lambda x : list(map(lambda y : mapping[y], list(x)))))
    
    mapping = {}
    vocab = [".", "(", ")"]
    for i, s in enumerate(vocab):
        mapping[s] = return_ohe(len(vocab), i)
    X_structure = np.stack(train["structure"].apply(lambda x : list(map(lambda y : mapping[y], list(x)))))
    
    
    X_node = np.concatenate([X_node, X_loop], axis = 2)
    
    ## interaction
    a = np.sum(X_node * (2 ** np.arange(X_node.shape[2])[None, None, :]), axis = 2)
    vocab = sorted(set(a.flatten()))
    print(vocab)
    ohes = []
    for v in vocab:
        ohes.append(a == v)
    ohes = np.stack(ohes, axis = 2)
    X_node = np.concatenate([X_node, ohes], axis = 2).astype(np.float32)
    
    
    print(X_node.shape)
    return X_node

X_node = get_input(train)
X_node_pub = get_input(test_pub)
X_node_pri = get_input(test_pri)

[np.int64(17), np.int64(18), np.int64(20), np.int64(24), np.int64(33), np.int64(34), np.int64(36), np.int64(40), np.int64(65), np.int64(66), np.int64(68), np.int64(72), np.int64(129), np.int64(130), np.int64(132), np.int64(136), np.int64(257), np.int64(258), np.int64(260), np.int64(264), np.int64(513), np.int64(514), np.int64(516), np.int64(520), np.int64(1025), np.int64(1026), np.int64(1028), np.int64(1032)]
(2096, 107, 39)
[np.int64(17), np.int64(18), np.int64(20), np.int64(24), np.int64(33), np.int64(34), np.int64(36), np.int64(40), np.int64(65), np.int64(66), np.int64(68), np.int64(72), np.int64(129), np.int64(130), np.int64(132), np.int64(136), np.int64(257), np.int64(258), np.int64(260), np.int64(264), np.int64(513), np.int64(514), np.int64(516), np.int64(520), np.int64(1025), np.int64(1026), np.int64(1028), np.int64(1032)]
(629, 107, 39)
[np.int64(17), np.int64(18), np.int64(20), np.int64(24), np.int64(33), np.int64(34), np.int64(36), np.int64(40), np.int64(65), np.int64(66), np

## model

In [11]:
import tensorflow as tf
from tensorflow.keras import layers as L
from tensorflow.keras import backend as K

def mcrmse(t, p, seq_len_target = seq_len_target):
    ## calculate mcrmse score by using numpy
    t = t[:, :seq_len_target]
    p = p[:, :seq_len_target]
    
    score = np.mean(np.sqrt(np.mean(np.mean((p - t) ** 2, axis = 1), axis = 0)))
    return score

def mcrmse_loss(t, y, seq_len_target = seq_len_target):
    ## calculate mcrmse score by using tf
    t = t[:, :seq_len_target]
    y = y[:, :seq_len_target]
    
    loss = tf.keras.ops.mean(tf.keras.ops.sqrt(tf.keras.ops.mean(tf.keras.ops.mean((t - y) ** 2, axis=1), axis=0)))
    return loss

def attention(x_inner, x_outer, n_factor, dropout):
    x_Q =  L.Conv1D(n_factor, 1, activation='linear', 
                  kernel_initializer='glorot_uniform',
                  bias_initializer='glorot_uniform',
                 )(x_inner)
    x_K =  L.Conv1D(n_factor, 1, activation='linear', 
                  kernel_initializer='glorot_uniform',
                  bias_initializer='glorot_uniform',
                 )(x_outer)
    x_V =  L.Conv1D(n_factor, 1, activation='linear', 
                  kernel_initializer='glorot_uniform',
                  bias_initializer='glorot_uniform',
                 )(x_outer)
    x_KT = L.Permute((2, 1))(x_K)
    res = L.Lambda(lambda c: K.batch_dot(c[0], c[1]) / np.sqrt(n_factor))([x_Q, x_KT])
#     res = tf.expand_dims(res, axis = 3)
#     res = L.Conv2D(16, 3, 1, padding = "same", activation = "relu")(res)
#     res = L.Conv2D(1, 3, 1, padding = "same", activation = "relu")(res)
#     res = tf.squeeze(res, axis = 3)
    att = L.Lambda(lambda c: K.softmax(c, axis=-1))(res)
    att = L.Lambda(lambda c: K.batch_dot(c[0], c[1]))([att, x_V])
    return att

def multi_head_attention(x, y, n_factor, n_head, dropout):
    if n_head == 1:
        att = attention(x, y, n_factor, dropout)
    else:
        n_factor_head = n_factor // n_head
        heads = [attention(x, y, n_factor_head, dropout) for i in range(n_head)]
        att = L.Concatenate()(heads)
        att = L.Dense(n_factor, 
                      kernel_initializer='glorot_uniform',
                      bias_initializer='glorot_uniform',
                     )(att)
    x = L.Add()([x, att])
    x = L.LayerNormalization()(x)
    if dropout > 0:
        x = L.Dropout(dropout)(x)
    return x

def res(x, unit, kernel = 3, rate = 0.1):
    h = L.Conv1D(unit, kernel, 1, padding = "same", activation = None)(x)
    h = L.LayerNormalization()(h)
    h = L.LeakyReLU()(h)
    h = L.Dropout(rate)(h)
    return L.Add()([x, h])

def forward(x, unit, kernel = 3, rate = 0.1):
#     h = L.Dense(unit, None)(x)
    h = L.Conv1D(unit, kernel, 1, padding = "same", activation = None)(x)
    h = L.LayerNormalization()(h)
    h = L.Dropout(rate)(h)
#         h = tf.keras.activations.swish(h)
    h = L.LeakyReLU()(h)
    h = res(h, unit, kernel, rate)
    return h

def adj_attn(x, adj, unit, n = 2, rate = 0.1):
    x_a = x
    x_as = []
    for i in range(n):
        x_a = forward(x_a, unit)
        x_a = tf.keras.ops.matmul(adj, x_a) ## aggregate neighborhoods
        x_as.append(x_a)
    if n == 1:
        x_a = x_as[0]
    else:
        x_a = L.Concatenate()(x_as)
    x_a = forward(x_a, unit)
    return x_a


def get_base(config):
    ## base model architecture 
    ## node, adj -> middle feature
    
    node = tf.keras.Input(shape = (None, X_node.shape[2]), name = "node")
    adj = tf.keras.Input(shape = (None, None, As.shape[3]), name = "adj")
    
    adj_learned = L.Dense(1, "relu")(adj)
    adj_all = L.Concatenate(axis = 3)([adj, adj_learned])
        
    xs = []
    xs.append(node)
    x1 = forward(node, 128, kernel = 3, rate = 0.0)
    x2 = forward(x1, 64, kernel = 6, rate = 0.0)
    x3 = forward(x2, 32, kernel = 15, rate = 0.0)
    x4 = forward(x3, 16, kernel = 30, rate = 0.0)
    x = L.Concatenate()([x1, x2, x3, x4])
    
    for unit in [64, 32]:
        x_as = []
        for i in range(adj_all.shape[3]):
            x_a = adj_attn(x, adj_all[:, :, :, i], unit, rate = 0.0)
            x_as.append(x_a)
        x_c = forward(x, unit, kernel = 30)
        
        x = L.Concatenate()(x_as + [x_c])
        x = forward(x, unit)
        x = multi_head_attention(x, x, unit, 4, 0.0)
        xs.append(x)
        
    x = L.Concatenate()(xs)

    model = tf.keras.Model(inputs = [node, adj], outputs = [x])
    return model


def get_ae_model(base, config):
    ## denoising auto encoder part
    ## node, adj -> middle feature -> node
    
    node = tf.keras.Input(shape = (None, X_node.shape[2]), name = "node")
    adj = tf.keras.Input(shape = (None, None, As.shape[3]), name = "adj")

    x = base([L.SpatialDropout1D(0.3)(node), adj])[0]
    x = forward(x, 64, rate = 0.3)
    p = L.Dense(X_node.shape[2], "sigmoid")(x)
    
    loss = L.Lambda(
        lambda z: -tf.keras.ops.mean(
            20 * z[0] * tf.keras.ops.log(z[1] + 1e-4)
            + (1 - z[0]) * tf.keras.ops.log(1 - z[1] + 1e-4), axis=(1, 2)
        ), name="denoising_loss"
    )([node, p])
    model = tf.keras.Model(inputs=[node, adj], outputs=[loss])
    
    opt = get_optimizer()
    model.compile(optimizer = opt, loss = lambda t, y : y)
    return model


def get_model(base, config):
    ## regression part
    ## node, adj -> middle feature -> prediction of targets
    
    node = tf.keras.Input(shape = (None, X_node.shape[2]), name = "node")
    adj = tf.keras.Input(shape = (None, None, As.shape[3]), name = "adj")
    
    x = base([node, adj])[0]
    x = forward(x, 128, rate = 0.4)
    x = L.Dense(5, None)(x)

    model = tf.keras.Model(inputs = [node, adj], outputs = [x])
    
    opt = get_optimizer()
    model.compile(optimizer = opt, loss = mcrmse_loss)
    return model

def get_optimizer():
#     sgd = tf.keras.optimizers.SGD(0.05, momentum = 0.9, nesterov=True)
    adam = tf.optimizers.Adam()
#     radam = tfa.optimizers.RectifiedAdam()
#     lookahead = tfa.optimizers.Lookahead(adam, sync_period=6)
#     swa = tfa.optimizers.SWA(adam)
    return adam

## pretrain

In [12]:
## here train denoising auto encoder model using all data

config = {} ## not use now
if ae_epochs > 0:
    base = get_base(config)
    ae_model = get_ae_model(base, config)
    ## TODO : simultaneous train
    for i in range(ae_epochs//ae_epochs_each):
        print(f"------ {i} ------")
        print("--- train ---")
        ae_model.fit([X_node, As], [X_node[:,0]],
                  epochs = ae_epochs_each,
                  batch_size = ae_batch_size)
        print("--- public ---")
        ae_model.fit([X_node_pub, As_pub], [X_node_pub[:,0]],
                  epochs = ae_epochs_each,
                  batch_size = ae_batch_size)
        print("--- private ---")
        ae_model.fit([X_node_pri, As_pri], [X_node_pri[:,0]],
                  epochs = ae_epochs_each,
                  batch_size = ae_batch_size)
        gc.collect()
    print("****** save ae model ******")
    base.save_weights("./base_ae.weights.h5")

------ 0 ------
--- train ---
Epoch 1/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 66s 331ms/step - loss: 0.9403
Epoch 2/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 25s 372ms/step - loss: 0.3278
Epoch 3/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 23s 341ms/step - loss: 0.1711
Epoch 4/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 25s 376ms/step - loss: 0.1122
Epoch 5/5
66/66 ━━━━━━━━━━━━━━━━━━━━ 23s 355ms/step - loss: 0.0843
--- public ---
Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 394ms/step - loss: 0.0665
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 467ms/step - loss: 0.0681
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 378ms/step - loss: 0.0595
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 470ms/step - loss: 0.0592
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 8s 409ms/step - loss: 0.0556
--- private ---
Epoch 1/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 64s 449ms/step - loss: 0.0552
Epoch 2/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 43s 459ms/step - loss: 0.0463
Epoch 3/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 42s 447ms/step - loss: 0.0360
Epoch 4/5
94/94 ━━━━━━━━━━━━━━━━━━━━ 42s 446ms/step - loss: 0.0371
Epoch

## train

In [13]:
## here train regression model from pretrain auto encoder model

from sklearn.model_selection import KFold
kfold = KFold(5, shuffle = True, random_state = 42)

scores = []
preds = np.zeros([len(X_node), X_node.shape[1], 5])
for i, (tr_idx, va_idx) in enumerate(kfold.split(X_node, As)):
    print(f"------ fold {i} start -----")
    print(f"------ fold {i} start -----")
    print(f"------ fold {i} start -----")
    X_node_tr = X_node[tr_idx]
    X_node_va = X_node[va_idx]
    As_tr = As[tr_idx]
    As_va = As[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]
    
    base = get_base(config)
    if ae_epochs > 0:
        print("****** load ae model ******")
        base.load_weights("./base_ae.weights.h5")
    model = get_model(base, config)
    if pretrain_dir is not None:
        d = f"./model{i}.weights.h5"
        print(f"--- load from {d} ---")
        model.load_weights(d)
    for epochs, batch_size in zip(epochs_list, batch_size_list):
        print(f"epochs : {epochs}, batch_size : {batch_size}")
        model.fit([X_node_tr, As_tr], [y_tr],
                  validation_data=([X_node_va, As_va], [y_va]),
                  epochs = epochs,
                  batch_size = batch_size, validation_freq = 3)
        
    model.save_weights(f"./model{i}.weights.h5")
    p = model.predict([X_node_va, As_va])
    scores.append(mcrmse(y_va, p))
    print(f"fold {i}: mcrmse {scores[-1]}")
    preds[va_idx] = p
    if one_fold:
        break
        
pd.to_pickle(preds, "oof.pkl")

------ fold 0 start -----
------ fold 0 start -----
------ fold 0 start -----
****** load ae model ******
epochs : 30, batch_size : 8
Epoch 1/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 61s 99ms/step - loss: 0.5836
Epoch 2/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 22s 105ms/step - loss: 0.3370
Epoch 3/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 26s 124ms/step - loss: 0.3039 - val_loss: 0.2794
Epoch 4/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 23s 111ms/step - loss: 0.2890
Epoch 5/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 23s 110ms/step - loss: 0.2797
Epoch 6/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 25s 121ms/step - loss: 0.2731 - val_loss: 0.2627
Epoch 7/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 24s 116ms/step - loss: 0.2649
Epoch 8/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 24s 116ms/step - loss: 0.2617
Epoch 9/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 26s 123ms/step - loss: 0.2564 - val_loss: 0.2569
Epoch 10/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 25s 118ms/step - loss: 0.2540
Epoch 11/30
210/210 ━━━━━━━━━━━━━━━━━━━━ 25s 119ms/step - loss: 0.2502
Epoch 12/30
210/210 ━━━━━━━━━━━━━━━━━━

In [14]:
print(scores)

[np.float64(0.22930383083127398), np.float64(0.22074591026541732), np.float64(0.23621101418002427), np.float64(0.23888142899774914), np.float64(0.23143612895084048)]


## predict

In [15]:
p_pub = 0
p_pri = 0
for i in range(5):
    model.load_weights(f"./model{i}.weights.h5")
    p_pub += model.predict([X_node_pub, As_pub]) / 5
    p_pri += model.predict([X_node_pri, As_pri]) / 5
    if one_fold:
        p_pub *= 5
        p_pri *= 5
        break

for i, target in enumerate(targets):
    test_pub[target] = [list(p_pub[k, :, i]) for k in range(p_pub.shape[0])]
    test_pri[target] = [list(p_pri[k, :, i]) for k in range(p_pri.shape[0])]

20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 12s 100ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 74ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 95ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 111ms/step
94/94 ━━━━━━━━━━━━━━━━━━━━ 9s 98ms/step


## sub

In [16]:
preds_ls = []
for df, preds in [(test_pub, p_pub), (test_pri, p_pri)]:
    for i, uid in enumerate(df.id):
        single_pred = preds[i]

        single_df = pd.DataFrame(single_pred, columns=targets)
        single_df['id_seqpos'] = [f'{uid}_{x}' for x in range(single_df.shape[0])]

        preds_ls.append(single_df)

preds_df = pd.concat(preds_ls)
preds_df.to_csv("submission.csv", index = False)
preds_df.head()

,reactivity,deg_Mg_pH10,deg_pH10,deg_Mg_50C,deg_50C,id_seqpos
0,0.777356,0.658518,1.855012,0.581316,0.793099,id_00073f8be_0
1,2.380903,3.244075,4.626174,3.304659,2.866361,id_00073f8be_1
2,1.628571,0.631524,0.864358,0.853824,0.823359,id_00073f8be_2
3,1.340907,1.055513,1.237771,1.592388,1.618518,id_00073f8be_3
4,0.811683,0.589177,0.632670,0.829958,0.814690,id_00073f8be_4


In [17]:
print(scores)
print(np.mean(scores))

[np.float64(0.22930383083127398), np.float64(0.22074591026541732), np.float64(0.23621101418002427), np.float64(0.23888142899774914), np.float64(0.23143612895084048)]
0.23131566264506104
